In [ ]:
# Importamos las libreias

# Importamos la librería Pandas,utilizamos alias como pd.
# Se utiliza para manipular y analizar datos en tablas (DataFrames).
import pandas as pd

# Importamos la librería sqlite3.
# Esta librería permite trabajar con bases de datos SQLite directamente desde Python.
import sqlite3

# Importamos la función create_engine desde SQLAlchemy.
# Esta función permite crear una conexión entre Python y una base de datos.
from sqlalchemy import create_engine
from sqlalchemy import text

engine = create_engine(('sqlite:///techventas.db'))

In [13]:
# Q1 - SELECT DISTINCT
# Elimina duplicados
# Lista todos los productos únicos disponibles usando un alias de columna descriptivo.
pd.read_sql("""
SELECT DISTINCT producto AS producto_unico
FROM ventas;
""", engine)

,producto_unico
0,Laptop Pro 15
1,Mouse Inalambrico
2,Monitor 27
3,Teclado Mecanico
4,SSD 1TB
5,Router WiFi 6
6,Desconocido


In [14]:
# Q2 - WHERE + BETWEEN
# Filtra por rango de fechas + condición
# Filtra pedidos del primer trimestre (ene–mar 2024) con cantidad mayor a 5 unidades.
pd.read_sql("""
SELECT *
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
  AND cantidad > 5;
""", engine)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1002,2024-01-07 00:00:00.000000,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
1,1004,2024-01-12 00:00:00.000000,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
2,1006,2024-01-18 00:00:00.000000,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20,95.0,0.00,Maria Torres
3,1008,2024-01-22 00:00:00.000000,C006,NetPrime,Norte,Router WiFi 6,Redes,8,120.0,0.00,Luis Mora
4,1009,2024-01-25 00:00:00.000000,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12,95.0,0.10,Maria Torres
5,1012,2024-02-05 00:00:00.000000,C008,BetaSoft,Centro,Mouse Inalambrico,Perifericos,30,25.0,0.05,Maria Torres
6,1015,2024-02-12 00:00:00.000000,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,25,95.0,0.10,Maria Torres
7,1017,2024-02-18 00:00:00.000000,C006,NetPrime,Norte,Monitor 27,Electronica,6,350.0,0.05,Luis Mora
8,1018,2024-02-20 00:00:00.000000,C002,Innovatek,Sur,SSD 1TB,Almacenamiento,18,95.0,0.00,Carlos Ruiz
9,1019,2024-02-22 00:00:00.000000,C007,AlphaTech,Sur,Mouse Inalambrico,Perifericos,12,25.0,0.00,Maria Torres


In [15]:
# Q3 - GROUP BY + HAVING
# Filtra después de agrupar (nivel agregado)
# Vendedores cuyo ingreso bruto total (cantidad × precio) supera $10,000 USD.
pd.read_sql("""
SELECT
    vendedor,
    SUM(cantidad * precio_unitario) AS ingreso_bruto_total
FROM ventas
GROUP BY vendedor
HAVING ingreso_bruto_total > 10000;
""", engine)

,vendedor,ingreso_bruto_total
0,Ana Lopez,30910.0
1,Carlos Ruiz,21355.0
2,Luis Mora,21930.0
3,Maria Torres,16410.0


In [16]:
# Q4 - COUNT + SUM + AVG
# Por categoría: total de pedidos, suma de unidades vendidas y precio unitario promedio.
pd.read_sql("""
SELECT
    categoria,
    COUNT(*) AS total_pedidos,
    SUM(cantidad) AS unidades_vendidas,
    AVG(precio_unitario) AS precio_promedio
FROM ventas
GROUP BY categoria;
""", engine)

,categoria,total_pedidos,unidades_vendidas,precio_promedio
0,Almacenamiento,12,260,95.0
1,Electronica,25,74,792.0
2,Perifericos,15,215,53.0
3,Redes,8,39,120.0


In [ ]:
# Q5 - JOIN
# Crear la tabla vendedores

with engine.begin() as conn:

    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS vendedores (
            vendedor TEXT PRIMARY KEY,
            zona TEXT,
            meta_mensual REAL
        )
    """))

    conn.execute(text("""
        INSERT OR REPLACE INTO vendedores VALUES
        ('Ana Lopez', 'Norte', 8000),
        ('Carlos Ruiz', 'Sur', 7500),
        ('Luis Mora', 'Norte', 8000),
        ('Maria Torres', 'Centro', 7000)
    """))

# Calcular el % de cumplimiento de la meta de cada vendedor

pd.read_sql("""
SELECT
    v.vendedor,
    v.zona,
    v.meta_mensual,
    SUM(t.cantidad * t.precio_unitario) AS venta_total,
    ROUND(
        (SUM(t.cantidad * t.precio_unitario) /
        v.meta_mensual) * 100, 2
    ) AS porcentaje_cumplimiento
FROM vendedores v
JOIN ventas t
    ON v.vendedor = t.vendedor
GROUP BY
    v.vendedor,
    v.zona,
    v.meta_mensual;
""", engine)

,vendedor,zona,meta_mensual,venta_total,porcentaje_cumplimiento
0,Ana Lopez,Norte,8000.0,30910.0,386.38
1,Carlos Ruiz,Sur,7500.0,21355.0,284.73
2,Luis Mora,Norte,8000.0,21930.0,274.13
3,Maria Torres,Centro,7000.0,16410.0,234.43


In [32]:
# Q6 - SUBCONSULTA
# Busca el máximo dentro de un agregado
# Encuentra el cliente que generó el mayor ingreso total en el primer semestre.
pd.read_sql("""
SELECT
    cliente_nombre,
    SUM(cantidad * precio_unitario) AS ingreso_total
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY cliente_nombre
HAVING ingreso_total = (
    SELECT MAX(total_ingreso)
    FROM (
        SELECT
            SUM(cantidad * precio_unitario) AS total_ingreso
        FROM ventas
        WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
        GROUP BY cliente_nombre
    )
);
""", engine)

,cliente_nombre,ingreso_total
0,TechCorp SA,20425.0
